# 00 · Carregamento e Preparação dos Dados

Lê o dataset **WikiIR 1k** dos arquivos locais em `data/wikIR1k/` e gera três arquivos:
- `data/corpus.jsonl` — documentos
- `data/queries.jsonl` — consultas (split de treino)
- `data/qrels.txt` — julgamentos de relevância (formato TREC)

## 1 · Instalação das dependências

In [7]:
# Execute apenas na primeira vez
! py -m pip install --upgrade pip
! py -m pip install pandas tqdm

## 2 · Imports e configuração

In [8]:
import json
from pathlib import Path

import pandas as pd
from tqdm import tqdm

WIKIIR_DIR   = Path('data/wikIR1k')
DATA_DIR     = Path('data')
DATA_DIR.mkdir(exist_ok=True)

CORPUS_PATH  = DATA_DIR / 'corpus.jsonl'
QUERIES_PATH = DATA_DIR / 'queries.jsonl'
QRELS_PATH   = DATA_DIR / 'qrels.txt'

print('Diretório WikiIR:', WIKIIR_DIR.resolve())

Diretório WikiIR: C:\Users\fsbertoglio\Documents\GitHub\automacao-lancamento-diario\dense-information-retrieval\data\wikIR1k


## 3 · Verificação dos arquivos locais

In [9]:
docs_csv    = WIKIIR_DIR / 'documents.csv'
queries_csv = WIKIIR_DIR / 'training' / 'queries.csv'
qrels_src   = WIKIIR_DIR / 'training' / 'qrels'

for p in [docs_csv, queries_csv, qrels_src]:
    status = 'OK' if p.exists() else 'NAO ENCONTRADO'
    print(f'  [{status}] {p}')

  [OK] data\wikIR1k\documents.csv
  [OK] data\wikIR1k\training\queries.csv
  [OK] data\wikIR1k\training\qrels


## 4 · Salvar o corpus

In [10]:
if CORPUS_PATH.exists() and CORPUS_PATH.stat().st_size > 0:
    print(f'[skip] {CORPUS_PATH} já existe.')
else:
    df = pd.read_csv(docs_csv)
    with open(CORPUS_PATH, 'w', encoding='utf-8') as f:
        for _, row in tqdm(df.iterrows(), total=len(df), desc='Documentos'):
            entry = {'doc_id': str(row['id_right']), 'text': str(row['text_right'])}
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')
    print(f'Corpus salvo → {CORPUS_PATH}  ({len(df):,} docs)')

Documentos: 100%|██████████| 369721/369721 [00:08<00:00, 43711.75it/s]

Corpus salvo → data\corpus.jsonl  (369,721 docs)


## 5 · Salvar as queries

In [11]:
if QUERIES_PATH.exists() and QUERIES_PATH.stat().st_size > 0:
    print(f'[skip] {QUERIES_PATH} já existe.')
else:
    df = pd.read_csv(queries_csv)
    with open(QUERIES_PATH, 'w', encoding='utf-8') as f:
        for _, row in tqdm(df.iterrows(), total=len(df), desc='Queries'):
            entry = {'query_id': str(row['id_left']), 'text': str(row['text_left'])}
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')
    print(f'Queries salvas → {QUERIES_PATH}  ({len(df):,} queries)')

Queries: 100%|██████████| 1444/1444 [00:00<00:00, 12573.88it/s]

Queries salvas → data\queries.jsonl  (1,444 queries)


## 6 · Salvar os qrels (formato TREC)

In [12]:
import shutil

if QRELS_PATH.exists() and QRELS_PATH.stat().st_size > 0:
    print(f'[skip] {QRELS_PATH} já existe.')
else:
    # O arquivo qrels já está no formato TREC (tab-separado)
    shutil.copy(qrels_src, QRELS_PATH)
    n = sum(1 for _ in open(QRELS_PATH))
    print(f'Qrels salvos → {QRELS_PATH}  ({n:,} linhas)')

Qrels salvos → data\qrels.txt  (47,699 linhas)


## 7 · Estatísticas e amostra

In [13]:
n_docs    = sum(1 for _ in open(CORPUS_PATH))
n_queries = sum(1 for _ in open(QUERIES_PATH))
n_qrels   = sum(1 for _ in open(QRELS_PATH))

print('── Estatísticas ──────────────────────────')
print(f'  Documentos : {n_docs:,}')
print(f'  Queries    : {n_queries:,}')
print(f'  Qrels      : {n_qrels:,}')

with open(CORPUS_PATH)  as f: sample_doc = json.loads(f.readline())
with open(QUERIES_PATH) as f: sample_q   = json.loads(f.readline())

print('\n── Amostra ───────────────────────────────')
print(f'  Doc   [{sample_doc["doc_id"]}]: {sample_doc["text"][:120]}...')
print(f'  Query [{sample_q["query_id"]}]: {sample_q["text"]}')
print('─' * 45)

── Estatísticas ──────────────────────────
  Documentos : 369,721
  Queries    : 1,444
  Qrels      : 47,699

── Amostra ───────────────────────────────
  Doc   [1781133]: it was used in landing craft during world war ii and is used today in private boats and training facilities the 6 71 is ...
  Query [123839]: yanni
─────────────────────────────────────────────
